# qkdpp vs. AIT-QKD: post-processing comparison

This notebook runs the same experimental dataset through two independent classical post-processing pipelines and compares the results:

- **[`qkdpp`](../README.md)** -- the Python package in this repository.
- **[AIT-QKD](https://github.com/axdhill/ait-qkd)** -- a C++ QKD software stack (modules: `enkey`, `error-estimation`, `cascade`, `confirmation`, `privacy-amplification`, `dekey`), run here inside a Docker container built from Debian 8 (the last release with the Qt4 toolchain this codebase requires).

Both pipelines do the same conceptual work -- parameter estimation, error correction, verification, privacy amplification -- but differ in two ways worth knowing about before reading the final numbers:

1. **Which error-rate estimate sizes privacy amplification.** AIT-QKD's `cascade` module writes the *post-correction* error rate (`corrected_bits / n`, i.e. with hindsight) into the key metadata, and `privacy-amplification` uses that. `qkdpp` uses the *public-sample* estimate from parameter estimation (no hindsight) -- see this repo's README for why.
2. **Fixed per-block overhead.** AIT-QKD processes data as discrete "keys" flowing through ZeroMQ pipes; here we chunk the experimental file into `CHUNK_BYTES`-sized pieces to feed it in, and privacy amplification's fixed security margin (`security_bits`) is paid once **per chunk**. `qkdpp` processes the whole string as one block by default, paying that fixed cost once. Fewer, larger chunks reduce this overhead on the AIT-QKD side; set `CHUNK_BYTES` to cover the whole file for a single-chunk run (see the note in section 3).

Neither difference makes one pipeline "correct" and the other "wrong" -- they reflect different design priorities. The point of this notebook is to make the comparison, and the reasons for any gap, explicit and reproducible.

**Requirements**: the AIT-QKD Docker image must already be built and this notebook run inside its JupyterLab (see the main project's Dockerfile). `qkdpp` alone does not need any of this -- see `notebooks/qkdpp_pipeline.ipynb` for a pipeline that only depends on this package.

In [ ]:
import sys, time, importlib
from pathlib import Path

sys.path.insert(0, "/work")
import qkdrun as q
importlib.reload(q)

work = Path("/work/run1")
work.mkdir(exist_ok=True, parents=True)

q.ensure_dbus()

## 1. Cleanup
Kills any leftover processes/state from a previous run. Always run this before a fresh full run.

In [ ]:
import subprocess, shutil

subprocess.run(["pkill", "-9", "-f", "/opt/qkd/build/bin/qkd-"], check=False)
subprocess.run(["pkill", "-9", "-f", "dbus-daemon --config-file"], check=False)
time.sleep(2)

shutil.rmtree("/tmp/qkd-test-dbus", ignore_errors=True)
shutil.rmtree("/tmp/qkd", ignore_errors=True)
for f in work.glob("*.log"):
    f.unlink()

q.ensure_dbus()
print("bus clean, modules registered now:", q.module_count())

## 2. Load the raw dataset and apply sifting

Sifting rule for this dataset (a phase-encoded, Sagnac-style protocol), validated against a known-good sifted pair (100% bit-for-bit match):

- `base_match == 1` -- Alice and Bob chose the same measurement basis
- `detectada == 1` -- a detection event occurred
- `detector != 'ambos'` -- excludes ambiguous double-click events

Alice's sifted bit is `bitA`. Bob's sifted bit is `bitA XOR error`.

If you use this notebook with a differently-structured CSV, the assertions below will fail loudly instead of silently producing a wrong sifted key -- adjust the rule for your schema.

In [ ]:
import pandas as pd
import numpy as np

csv_path = work / "rondas_completas.csv"
assert csv_path.exists(), f"couldn't find {csv_path} -- check the file name/location"

raw = pd.read_csv(csv_path)
print(f"raw rounds: {len(raw)}")

required_cols = {"round_id", "base_match", "detectada", "detector", "bitA", "error"}
missing = required_cols - set(raw.columns)
assert not missing, f"expected columns missing: {missing}"

mask = (raw.base_match == 1) & (raw.detectada == 1) & (raw.detector != "ambos")
sifted = raw[mask].sort_values("round_id")
assert sifted["error"].notna().all(), "sifting selected rows with no error flag -- wrong rule for this dataset"

alice_bits = sifted["bitA"].astype(int).to_numpy()
bob_bits = (sifted["bitA"].astype(int) ^ sifted["error"].astype(int)).to_numpy()

n_sifted = len(alice_bits)
qber_true = float(np.mean(alice_bits != bob_bits))
print(f"sifted bits: {n_sifted}")
print(f"true QBER (both sides known -- for diagnostics only): {qber_true:.4f}")

alice_txt = work / "alice_sifted.txt"
bob_txt = work / "bob_sifted.txt"
alice_txt.write_text("".join(map(str, alice_bits.tolist())))
bob_txt.write_text("".join(map(str, bob_bits.tolist())))
print(f"saved: {alice_txt.name}, {bob_txt.name}")

## 3. Post-processing via AIT-QKD

`enkey -> error-estimation (PE) -> cascade (EC) -> confirmation -> privacy-amplification (PA) -> dekey`

`CHUNK_BYTES` controls how many AIT-QKD "keys" the experimental file is split into (`enkey.key_size`). Set it to cover the whole sifted string (`n_sifted // 8`) for a single chunk -- avoids paying `security_bits` more than once. The value below (128) instead reproduces a multi-chunk run for comparison; see this notebook's write-up (or the repo's conversation log) for the efficiency trade-off between the two choices.

In [ ]:
CHUNK_BYTES = 128    # bytes per AIT-QKD "key"; set to n_sifted // 8 for a single chunk
PE_DISCLOSE = 0.1    # fraction publicly revealed to estimate QBER (same convention as qkdpp)
PA_SECURITY_BITS = 100   # AIT-QKD's fixed PA security margin; ~ qkdpp's pa_cost, paid once per chunk

a_bits = alice_bits.astype(np.uint8)
b_bits = bob_bits.astype(np.uint8)

n_bytes = (n_sifted // 8 // CHUNK_BYTES) * CHUNK_BYTES
n_bits_used = n_bytes * 8
n_chunks = n_bytes // CHUNK_BYTES
a_bits_ait, b_bits_ait = a_bits[:n_bits_used], b_bits[:n_bits_used]
print(f"AIT-QKD: using {n_bits_used} of {n_sifted} sifted bits ({n_chunks} chunks of "
      f"{CHUNK_BYTES}B; {n_sifted - n_bits_used} tail bits dropped for not filling a full chunk)")

alice_bin = work / "alice_packed.bin"
bob_bin = work / "bob_packed.bin"
np.packbits(a_bits_ait).tofile(alice_bin)
np.packbits(b_bits_ait).tofile(bob_bin)

ait_config = f"""[module]

enkey.key_size = {CHUNK_BYTES}
enkey.alice.file_url = file://{alice_bin.resolve()}
enkey.alice.url_pipe_out = ipc:///tmp/qkd/ee.alice.in
enkey.bob.file_url = file://{bob_bin.resolve()}
enkey.bob.url_pipe_out = ipc:///tmp/qkd/ee.bob.in

error-estimation.alice.url_peer = tcp://127.0.0.1:7140
error-estimation.alice.url_pipe_in = ipc:///tmp/qkd/ee.alice.in
error-estimation.alice.url_pipe_out = ipc:///tmp/qkd/cascade.alice.in
error-estimation.bob.url_listen = tcp://127.0.0.1:7140
error-estimation.bob.url_pipe_in = ipc:///tmp/qkd/ee.bob.in
error-estimation.bob.url_pipe_out = ipc:///tmp/qkd/cascade.bob.in
error-estimation.disclose = {PE_DISCLOSE}

cascade.alice.url_peer = tcp://127.0.0.1:7130
cascade.alice.url_pipe_in = ipc:///tmp/qkd/cascade.alice.in
cascade.alice.url_pipe_out = ipc:///tmp/qkd/confirmation.alice.in
cascade.bob.url_listen = tcp://127.0.0.1:7130
cascade.bob.url_pipe_in = ipc:///tmp/qkd/cascade.bob.in
cascade.bob.url_pipe_out = ipc:///tmp/qkd/confirmation.bob.in
cascade.passes = 14

confirmation.alice.url_peer = tcp://127.0.0.1:7160
confirmation.alice.url_pipe_in = ipc:///tmp/qkd/confirmation.alice.in
confirmation.alice.url_pipe_out = ipc:///tmp/qkd/pa.alice.in
confirmation.bob.url_listen = tcp://127.0.0.1:7160
confirmation.bob.url_pipe_in = ipc:///tmp/qkd/confirmation.bob.in
confirmation.bob.url_pipe_out = ipc:///tmp/qkd/pa.bob.in
confirmation.rounds = 10

privacy-amplification.alice.url_peer = tcp://127.0.0.1:7180
privacy-amplification.alice.url_pipe_in = ipc:///tmp/qkd/pa.alice.in
privacy-amplification.alice.url_pipe_out = ipc:///tmp/qkd/dekey.alice.in
privacy-amplification.bob.url_listen = tcp://127.0.0.1:7180
privacy-amplification.bob.url_pipe_in = ipc:///tmp/qkd/pa.bob.in
privacy-amplification.bob.url_pipe_out = ipc:///tmp/qkd/dekey.bob.in
privacy-amplification.security_bits = {PA_SECURITY_BITS}

dekey.terminate_after = {n_chunks}
dekey.alice.url_pipe_in = ipc:///tmp/qkd/dekey.alice.in
dekey.alice.file_url = file://{(work / 'alice_final_ait.bin').resolve()}
dekey.bob.url_pipe_in = ipc:///tmp/qkd/dekey.bob.in
dekey.bob.file_url = file://{(work / 'bob_final_ait.bin').resolve()}
"""
config_ait = work / "pipeline_ait.conf"
config_ait.write_text(ait_config)

# terminate_after is set only on the LAST module (dekey): setting it on every module in the
# chain causes a ZeroMQ message-loss race during shutdown (queued messages get dropped when
# an upstream module exits before delivery is confirmed).
procs = []
for module in ["dekey", "privacy-amplification", "confirmation", "cascade", "error-estimation", "enkey"]:
    procs += q.launch(module, config_ait, work)

ok = q.wait_for_modules(12, timeout=30)
print("modules:", q.module_count(), "| ok:", ok)

In [ ]:
for i in range(60):
    time.sleep(1)
    n_registered = q.module_count()
    if i % 5 == 0:
        print(f"t={i}s  active modules: {n_registered}")
    p = work / "bob_final_ait.bin"
    if p.exists() and p.stat().st_size > 0:
        print("dekey has written and closed the file")
        time.sleep(2)
        break

q.kill_all(procs)

print("\n--- AIT-QKD final files ---")
for name in ["alice_final_ait.bin", "bob_final_ait.bin"]:
    p = work / name
    print(name, p.stat().st_size if p.exists() else "MISSING", "bytes")

## 4. Post-processing via qkdpp

`sifting.estimate_qber -> cascade.reconcile -> extract.verify -> extract.amplify`, wired together by `qkdpp.run(...)`. Uses the same public-sample fraction (`PE_DISCLOSE`) as the AIT-QKD run above, on the whole sifted string as a single block.

In [ ]:
import qkdpp
importlib.reload(qkdpp)

a_full = alice_bits.astype(np.uint8)
b_full = bob_bits.astype(np.uint8)

r = qkdpp.run(a_full, b_full, pe_fraction=PE_DISCLOSE, n_passes=10, seed=1)
print(r.summary())

qkdpp.io.save_bits(work / "bob_final_qkdpp.txt", r.key)
print("final key saved to", work / "bob_final_qkdpp.txt")

## 5. Side-by-side comparison

**A direct bit-for-bit comparison between the two final keys is not meaningful** -- they are independent secrets from two different pipelines (different error-rate estimate, different random Toeplitz seed), so ~50% agreement is the *expected*, correct outcome, not a sign of a bug.

What the two lengths below actually reflect: how much of the sifted string each pipeline could turn into final key, given its own error-rate estimate and chunking. See the introduction at the top of this notebook for why they differ.

In [ ]:
ait_final_len = (work / "bob_final_ait.bin").stat().st_size * 8 if (work / "bob_final_ait.bin").exists() else 0
qkdpp_final_len = r.final_len

print(f"{'':20s} {'final bits':>12s}")
print(f"{'AIT-QKD (with PA)':20s} {ait_final_len:>12d}")
print(f"{'qkdpp (with PA)':20s} {qkdpp_final_len:>12d}")
print()
print(f"true QBER of the raw data:            {qber_true:.4f}")
print(f"QBER estimated by qkdpp's PE sample:   {r.qber:.4f}")